# GRPO Training for Function-Calling (Variants 4-6)

GRPO on Qwen2.5-3B-Instruct with QLoRA.
- **V4**: cold-start, unmasked data (`xlam_4k_formatted.jsonl`)
- **V5**: cold-start, masked data (`xlam_4k_masked_formatted.jsonl`)
- **V6**: SFT-init from V3 masked adapter, then GRPO with masked + curriculum-sorted data

**Runtime**: A100 GPU required. ~3-4h per cold-start variant. V6 is similar; loading the V3 adapter does not change wall clock materially.

**V6 prerequisite**: `arnursb/info290-v3-sft-masked` must exist on HF Hub before running the V6 cell.


In [15]:
!cd /content/repo && git pull --ff-only

remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 1.01 KiB | 1.01 MiB/s, done.
From https://github.com/arnursabet/info290-function-calling-slm
   a49dca0..9250d47  main       -> origin/main
Updating a49dca0..9250d47
Fast-forward
 configs/grpo_curriculum.yaml | 1 +
 src/training/grpo.py         | 2 ++
 2 files changed, 3 insertions(+)


In [2]:
from google.colab import drive, userdata
drive.mount("/content/drive")

# Clone repo fresh; the prior path /content/drive/MyDrive/info290-genai-spring2026/final-project
# does not exist on Arnur's Drive. Use the same pattern as sft_training.ipynb.
import os
gh_token = userdata.get("GH_TOKEN")
!git clone https://{gh_token}@github.com/arnursabet/info290-function-calling-slm.git /content/repo
%cd /content/repo

!pip install -q torch transformers==5.2.0 datasets==4.5.0 trl==0.27.1 \
    peft==0.18.1 accelerate bitsandbytes wandb pyyaml

Mounted at /content/drive
Cloning into '/content/repo'...
remote: Enumerating objects: 717, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 717 (delta 134), reused 139 (delta 82), pack-reused 485 (from 1)
Receiving objects: 100% (717/717), 6.17 MiB | 27.59 MiB/s, done.
Resolving deltas: 100% (401/401), done.
/content/repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.7 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
import os

# HF token must be in Colab secrets. Required for adapter push to Hub.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Hard-fail on missing W&B key. Silent fallback would log to whoever's
# creds are cached in ~/.netrc, which is not always the project owner.
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception as e:
    raise RuntimeError(
        "WANDB_API_KEY not in Colab secrets. Get it at https://wandb.ai/authorize, "
        "then add via the key icon in the left sidebar."
    ) from e

# Force wandb to use THIS key, not whatever ~/.netrc had cached from a
# previous Colab account. Without --relogin, wandb keeps the stale cache.
!wandb login --relogin --key $WANDB_API_KEY

# Fail fast if either identity is wrong, before launching 3-5h of training.
from huggingface_hub import HfApi
import wandb

hf_user = HfApi().whoami()["name"]
wandb_user = wandb.Api().viewer.username

print(f"HF authenticated as     {hf_user}")
print(f"WANDB authenticated as  {wandb_user}")

EXPECTED_HF = "arnursb"
if hf_user != EXPECTED_HF:
    raise RuntimeError(f"Expected HF user {EXPECTED_HF!r}, got {hf_user!r}")
print("Identity check passed - safe to train.")


Usage: wandb login [OPTIONS] [KEY]...
Try 'wandb login --help' for help.

Error: No such option: --key Did you mean --verify?


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


HF authenticated as     arnursb
WANDB authenticated as  sabet
Identity check passed - safe to train.


In [4]:
from pathlib import Path

# Generate data if missing (the GRPO notebook can be run independently from SFT).
if not Path("data/processed/xlam_4k_formatted.jsonl").exists():
    !python -m src.data.download --output data/raw/xlam_60k.json
    !python -m src.data.filter --input data/raw/xlam_60k.json --output data/processed/xlam_4k.json
    !python -m src.data.mask --input data/processed/xlam_4k.json --output data/processed/xlam_4k_masked.json
    !python -m src.data.format --input data/processed/xlam_4k.json --output data/processed/xlam_4k_formatted.jsonl
    !python -m src.data.format --input data/processed/xlam_4k_masked.json --output data/processed/xlam_4k_masked_formatted.jsonl

# V6 needs a curriculum-sorted variant of the masked formatted JSONL.
data_v6 = Path("data/processed/xlam_4k_masked_curriculum_formatted.jsonl")
if not data_v6.exists():
    !python -m src.data.curriculum \
        --input data/processed/xlam_4k_masked_formatted.jsonl \
        --output data/processed/xlam_4k_masked_curriculum_formatted.jsonl

data_v4 = Path("data/processed/xlam_4k_formatted.jsonl")
data_v5 = Path("data/processed/xlam_4k_masked_formatted.jsonl")
print(f"V4 data: {data_v4.exists()} ({sum(1 for _ in open(data_v4)) if data_v4.exists() else 0} examples)")
print(f"V5 data: {data_v5.exists()} ({sum(1 for _ in open(data_v5)) if data_v5.exists() else 0} examples)")
print(f"V6 data: {data_v6.exists()} ({sum(1 for _ in open(data_v6)) if data_v6.exists() else 0} examples, curriculum-sorted)")


README.md: 10.7kB [00:00, 18.3MB/s]
xlam_function_calling_60k.json: 100% 96.1M/96.1M [00:01<00:00, 58.5MB/s]
Generating train split: 100% 60000/60000 [00:00<00:00, 69571.24 examples/s]
Downloaded 60000 examples
Saved to data/raw/xlam_60k.json

Total examples: 60000

-- Tools per example --
  1 tools: 15410
  2 tools: 12726
  3 tools: 12655
  4 tools: 13078
  5 tools: 2427
  6 tools: 1464
  7 tools: 1108
  8 tools: 1132

-- Function calls per example --
  1 calls: 28461
  2 calls: 25422
  3 calls: 4697
  4 calls: 1056
  5 calls: 187
  6 calls: 94
  7 calls: 20
  8 calls: 12
  9 calls: 12
  10 calls: 19
  11 calls: 4
  12 calls: 6
  14 calls: 2
  19 calls: 1
  20 calls: 2
  23 calls: 1
  24 calls: 1
  26 calls: 1
  30 calls: 1
  52 calls: 1

-- Call type breakdown --
  parallel_calls: 31539 (52.6%)
  single_call: 28461 (47.4%)

Avg parameters per call: 1.7
Max parameters in a single call: 19
Valid examples: 60000 / 60000
After dedup: 58148
  single: 26735
  parallel: 31413
  no_call: 0



## Smoke test (run before V4-V6)

Cap GRPO at 5 optimizer steps with `--max-steps 5 --no-push`. Exercises the
full pipeline (model load, PEFT wrap, rollout, reward, optim, save) in ~3 min
on an A100. If this fails, V4-V6 will fail the same way - patch first.

In [11]:
# Smoke: 5 optimizer steps, no Hub push. ~3 min on A100.
# If this completes, V4/V5/V6 will train cleanly.
!python -m src.training.grpo \
    --config configs/grpo.yaml \
    --data data/processed/xlam_4k_formatted.jsonl \
    --run-name grpo-smoke \
    --no-push \
    --max-steps 5

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sabet (sabet-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/repo/wandb/run-20260430_052604-8ldl20eh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-smoke
wandb: ⭐️ View project at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm
wandb: 🚀 View run at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm/runs/8ldl20eh
Dataset: 4000 examples
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:02<00:00, 202.30it/s, Materializing param=model.norm.weight]
warmup_ratio is deprecated and will be remo

In [8]:
!python -m src.training.grpo \
    --config configs/grpo.yaml \
    --data data/processed/xlam_4k_formatted.jsonl \
    --run-name grpo-v4 \
    --hf-repo-id arnursb/info290-v4-grpo

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sabet (sabet-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/repo/wandb/run-20260430_042511-h9boab78
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-v4
wandb: ⭐️ View project at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm
wandb: 🚀 View run at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm/runs/h9boab78
Dataset: 4000 examples
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:02<00:00, 201.67it/s, Materializing param=model.norm.weight]
warmup_ra

In [6]:
!python -m src.training.grpo \
    --config configs/grpo.yaml \
    --data data/processed/xlam_4k_masked_formatted.jsonl \
    --run-name grpo-v5 \
    --hf-repo-id arnursb/info290-v5-grpo-masked

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sabet (sabet-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run cam42sul (0.0s)
wandb: ⣻ setting up run cam42sul (0.0s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/repo/wandb/run-20260430_031505-cam42sul
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-v5
wandb: ⭐️ View project at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm
wandb: 🚀 View run at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm/runs/cam42sul
Dataset: 4000 examples
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  81% 352/434 [07:09<01:40,  1.22s/it, Materializing param=model.layers.29.mlp.gate_proj.weight]
Traceback (most recent call last):
  File "<frozen runpy>",

In [22]:
!python -m src.training.grpo \
  --config configs/grpo_curriculum.yaml \
  --data data/processed/xlam_4k_masked_formatted.jsonl \
  --run-name grpo-v5-coldstart-masked \
  --hf-repo-id arnursb/info290-v5-grpo-masked \
  --max-steps 400

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sabet (sabet-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/repo/wandb/run-20260430_070525-hz3n0h95
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-v5-coldstart-masked
wandb: ⭐️ View project at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm
wandb: 🚀 View run at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm/runs/hz3n0h95
Dataset: 4000 examples
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:02<00:00, 199.51it/s, Materializing param=model.norm.weight]
warmup_ratio is deprecated an

## V6: Full recipe (SFT-init + GRPO + masked + curriculum)

Loads `arnursb/info290-v3-sft-masked` as starting weights via `--init-adapter`, then runs GRPO with `configs/grpo_curriculum.yaml` on curriculum-sorted masked data. Pushes to `arnursb/info290-v6-grpo-curriculum`.

Run only after V3 SFT-masked has finished and `adapter_config.json` is visible on Hub.


In [21]:
!python -m src.training.grpo \
    --config configs/grpo_curriculum.yaml \
    --data data/processed/xlam_4k_formatted.jsonl \
    --run-name grpo-v6-coldstart \
    --hf-repo-id arnursb/info290-v6-grpo-curriculum \
    --max-steps 400
# !python -m src.training.grpo \                                                                                                          --config configs/grpo_curriculum.yaml \
#   --data data/processed/xlam_4k_formatted.jsonl \
#   --run-name grpo-v6-coldstart \
#   --hf-repo-id arnursb/info290-v6-grpo-curriculum \
#   --max-steps 400

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sabet (sabet-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/repo/wandb/run-20260430_060102-aegu59in
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-v6-coldstart
wandb: ⭐️ View project at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm
wandb: 🚀 View run at https://wandb.ai/sabet-university-of-california-berkeley/info290-function-calling-slm/runs/aegu59in
Dataset: 4000 examples
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:02<00:00, 202.76it/s, Materializing param=model.norm.weight]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps

In [8]:
import os, shutil

# HF push is the canonical save (--hf-repo-id in cells 4/5). Drive is redundant backup.
src = "checkpoints/grpo"
dst = "/content/drive/MyDrive/info290-slm/checkpoints/grpo"
os.makedirs(dst, exist_ok=True)
if os.path.exists(src):
    for variant in os.listdir(src):
        src_v = os.path.join(src, variant)
        dst_v = os.path.join(dst, variant)
        if os.path.exists(dst_v):
            print(f"{variant}: already on Drive, skipping")
            continue
        shutil.copytree(src_v, dst_v)
        assert os.path.exists(dst_v) and os.listdir(dst_v), f"Drive copy failed for {variant}"
        print(f"{variant}: copied to {dst_v}")
else:
    print("No GRPO checkpoints found locally.")

No GRPO checkpoints found locally.


In [ ]:
!nvidia-smi